# Demo 06 - which devices need attention today

The dashboard does not ask "what is in the data", it asks "what should someone look at".

Thresholds are **not** stored in bronze. A threshold is a decision, not a fact, and decisions
change: on these dev boxes 90 percent disk usage is the intended state, because they hold a full
month of recordings on fewer disks than production. The same reading on a production device would
be a phone call at 2am. If the flag were written into the row, you would need two copies of the
same data.

So bronze keeps what arrived, and a view on top computes flags at read time. That view is really
the beginning of a silver layer; these labs stop at bronze, so it lives here.

Thresholds are widgets and can be changed while the dashboard is on screen.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("disk_temp_max", "50")
dbutils.widgets.text("cpu_max", "80")
dbutils.widgets.text("disk_used_max", "95")
dbutils.widgets.text("gap_minutes", "120")
dbutils.widgets.text("focus_device", "NAS6CC246")

login        = dbutils.widgets.get("login")
catalog      = dbutils.widgets.get("target_catalog")
temp_max     = float(dbutils.widgets.get("disk_temp_max"))
cpu_max      = float(dbutils.widgets.get("cpu_max"))
disk_max     = float(dbutils.widgets.get("disk_used_max"))
gap_min      = int(dbutils.widgets.get("gap_minutes"))
focus_device = dbutils.widgets.get("focus_device")
assert all([login, catalog])

demo     = f"{login}_demo_bronze"
readings = f"{catalog}.{demo}.sensor_readings_bronze"
registry = f"{catalog}.{demo}.device_registry_bronze"
flagged  = f"{catalog}.{demo}.v_readings_flagged"

## The rules

Six of them. Four are thresholds on a value, and the last two are the interesting ones because they
are about data that is missing rather than data that is wrong:

- `incomplete_reading` - the device answered but returned no `system_stats`, so there is no name, no
  CPU, no memory. Around 900 of these. They still carry disk temperatures, which is why throwing
  them away at load time would have lost real measurements.
- `reporting_gap` - too long since that device's previous reading. No threshold on a value catches a
  device that simply went quiet.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {flagged} AS
WITH r AS (
  SELECT *,
         to_timestamp(reading_ts) AS ts,
         lag(to_timestamp(reading_ts)) OVER (
             PARTITION BY device_id ORDER BY to_timestamp(reading_ts)) AS prev_ts
  FROM {readings}
)
SELECT *,
       date(ts)                                            AS reading_date,
       disk_temp_c   > {temp_max}                          AS hot_disk,
       cpu_usage_pct > {cpu_max}                           AS cpu_pegged,
       disk_used_pct > {disk_max}                          AS disk_filling,
       system_health = 'alert'                             AS health_alert,
       device_id IS NULL                                   AS incomplete_reading,
       prev_ts IS NOT NULL
         AND timestampdiff(MINUTE, prev_ts, ts) > {gap_min} AS reporting_gap
FROM r
""")

RULES = ["hot_disk", "cpu_pegged", "disk_filling",
         "health_alert", "incomplete_reading", "reporting_gap"]

any_flag = F.expr(" OR ".join(f"coalesce({r}, false)" for r in RULES))
flags    = spark.table(flagged).withColumn("any_flag", any_flag)

print(f"thresholds: disk {temp_max}C, cpu {cpu_max}%, disk used {disk_max}%, gap {gap_min} min")

## Panel 1 - how much is wrong

In [0]:
display(flags.agg(
    F.count("*").alias("readings"),
    F.countDistinct("device_id").alias("devices"),
    F.sum(F.col("any_flag").cast("int")).alias("flagged"),
    F.round(F.avg(F.col("any_flag").cast("int")) * 100, 1).alias("flagged_pct"),
    F.min("reading_date").alias("from"),
    F.max("reading_date").alias("to")))

Databricks visualization. Run in Databricks to view.

## Panel 2 - which device

Sorted worst first. The list someone would actually work through in the morning.


In [0]:
display(flags.groupBy("device_id", "source_system").agg(
    F.count("*").alias("readings"),
    F.sum(F.col("any_flag").cast("int")).alias("flagged"),
    F.round(F.avg(F.col("any_flag").cast("int")) * 100, 1).alias("flagged_pct"),
    F.round(F.avg("disk_temp_c"), 1).alias("avg_disk_temp"),
    F.max("disk_temp_c").alias("max_disk_temp"),
    F.round(F.avg("cpu_usage_pct"), 1).alias("avg_cpu"),
).orderBy(F.desc("flagged")))

Databricks visualization. Run in Databricks to view.

## Panel 3 - which rule fired

A hot disk and a device that went quiet are both "a problem", but not the same problem and not the
same person's job.

In [0]:
by_rule = flags.agg(*[F.sum(F.col(r).cast("int")).alias(r) for r in RULES]).first().asDict()
display(spark.createDataFrame(
    [(k, int(v or 0)) for k, v in by_rule.items()], "rule string, hits int"
).orderBy(F.desc("hits")))

Databricks visualization. Run in Databricks to view.

## Panel 4 - disk temperature over time

The one panel that would justify the whole system on its own. A device whose temperature climbs
over weeks is a disk about to take a month of recordings with it.

In [0]:
display(flags
        .filter(F.col("device_id").isNotNull())
        .groupBy("reading_date", "device_id")
        .agg(F.round(F.avg("disk_temp_c"), 1).alias("avg_disk_temp"),
             F.max("disk_temp_c").alias("max_disk_temp"),
             F.round(F.avg("cpu_usage_pct"), 1).alias("avg_cpu"))
        .orderBy("reading_date", "device_id"))

Databricks visualization. Run in Databricks to view.

## Panel 5 - health status over time

The device does not go from fine to broken. It reports `good`, then `warning`, and only later
`alert`. In this data the alerts start partway through the period, which is exactly the kind of
thing you cannot see without keeping history.


In [0]:
# one device only. Degradation is something a single box goes through, and three
# overlapping histories on one chart just cancel each other out
display(flags
        .filter(F.col("device_id") == focus_device)
        .filter(F.col("system_health").isNotNull())
        .groupBy("reading_date", "system_health")
        .agg(F.count("*").alias("readings"))
        .orderBy("reading_date", "system_health"))

Databricks visualization. Run in Databricks to view.

## The new device

Two columns exist only for the box that reports them. Everything older has null there, and nothing
had to be rewritten for that to be true.


In [0]:
cols = spark.table(readings).columns
if "psu_temp_c" in cols:
    display(flags.groupBy("device_id").agg(
        F.count("*").alias("readings"),
        F.count("psu_temp_c").alias("psu_temp_reported"),
        F.round(F.avg("psu_temp_c"), 1).alias("avg_psu_temp"),
        F.round(F.avg("fan_rpm"), 0).alias("avg_fan_rpm"),
    ).orderBy("device_id"))
else:
    print("no psu_temp_c column yet - run demo_04 part 2 and the job")

## Where the cameras are

The registry is what turns a device id into somewhere a person can walk to. A recorder with a rising
temperature matters more when 26 cameras across 13 locations depend on it.

Watch for the device that is **missing** from this list: it has readings but no registry entry,
because the hardware was plugged in before anyone wrote it down. That is a real onboarding gap and
the dashboard just found it.

Camera side note: the object labels and camera statuses in the Lab 1 data were drawn from weighted
distributions, so they show a plausible spread but no real cause. The NAS telemetry above is genuine.

In [0]:
device_flags = (flags.filter(F.col("device_id").isNotNull())
                .groupBy("device_id")
                .agg(F.sum(F.col("any_flag").cast("int")).alias("flagged")))

cameras = (spark.table(registry)
           .groupBy("device_id")
           .agg(F.count("*").alias("cameras"),
                F.countDistinct("location").alias("locations")))

# full outer, so a device with readings but no registry entry is visible instead of dropped
display(device_flags.join(cameras, "device_id", "full").orderBy(F.desc("flagged")))

## From a number back to a file

The last step of any dashboard nobody trusts yet: pick a flagged reading and show where it came
from. `source_system` says which way it arrived, `source_file` names the file for the batch path,
`eh_partition` and `eh_offset` do the same job for the stream, and `ingestion_ts` says when we
processed it.

Ten rows from each path, so both sets of metadata are on screen at once.

In [0]:
from pyspark.sql import Window

per_source = Window.partitionBy("source_system").orderBy(F.desc("reading_ts"))

display(flags.filter(F.col("any_flag"))
        .withColumn("rn", F.row_number().over(per_source))
        .filter(F.col("rn") <= 10)
        .select("device_id", "reading_ts", "disk_temp_c", "system_health",
                "hot_disk", "health_alert", "reporting_gap",
                "source_system", "source_file", "eh_partition", "eh_offset", "ingestion_ts")
        .orderBy("source_system", F.desc("reading_ts")))